In [ ]:
from itertools import cycle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, DistributedSampler
from torchvision import datasets, transforms
from model import LeNet5


tran = transforms.Compose((transforms.ToTensor(), transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))))
full_train_dataset = datasets.CIFAR100(root='./data', train=True, download=True, transform=tran)

train_size = int(0.8 * len(full_train_dataset))  # 80% for training
val_size = len(full_train_dataset) - train_size  # 20% for validation

train_dataset, val_dataset = random_split(full_train_dataset, (train_size, val_size))

# dloader = DataLoader(train_dataset, 10000, shuffle=True, pin_memory=True)
# print(len(dloader))


In [ ]:
from distributed import distributed_learning

model = distributed_learning(train_dataset, 5, 2, 2, 500, torch.optim.Adam, {"lr": 0.01})

In [ ]:

class PeekableIter:
    def __init__(self, iterable):
        self.iterator = iter(iterable)
        self._next_item = None

    def _advance(self):
        try:
            self._next_item = next(self.iterator)
        except StopIteration:
            self._next_item = None

    def peek(self):
        if self._next_item is None:
            self._advance()
        return self._next_item

    def next(self):
        if self._next_item is None:
            self._advance()
        result = self._next_item
        self._advance()
        return result


class DistributedDataLoader:

    def __init__(self, data, n_workers, local_batch_size):
        self.data = data
        self.n_workers = n_workers
        self.batch_size = local_batch_size

        self.shuffle()

    def shuffle(self):
        self.loaders= [PeekableIter(DataLoader(d, self.batch_size, shuffle=False)) for d in split_N_ways(self.data, self.n_workers)]
        self.need_shuffle: torch.Tensor = torch.zeros(self.n_workers, dtype=bool)

    def set_current_worker(self, current_id):
        self.current_id = current_id

    def get_next_batch(self, id):
        if self.need_shuffle.all():
            self.shuffle()
        elif self.need_shuffle[id]:
            raise Exception("Finish epochs on all workers first") 

        this_batch = self.loaders[id].next()

        next_batch = self.loaders[id].peek()

        if next_batch is None:
            self.need_shuffle[id] = True

        return this_batch
    
    def __next__(self):
        return self.get_next_batch(self.current_id)



In [ ]:
from typing import Iterator
from torch.utils.data import Dataset

class SimpleNumberDataset(Dataset):
    def __init__(self):
        # Input data: numbers from 1 to 10
        self.data = torch.arange(100, dtype=torch.float32)
        self.labels = torch.arange(100, dtype=torch.long) 

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index], self.labels[index]


def shuffle_and_split(train_data, N, random_seed=None) -> list[Iterator[DataLoader]]:
    if random_seed is not None:
        random_seed = torch.Generator().manual_seed(random_seed)

    lengths = [len(train_data) // N] * N

    return [iter(DataLoader(subset, batch_size=1)) for subset in random_split(train_data, lengths, random_seed)]

d = SimpleNumberDataset()

a = shuffle_and_split(d, 5)



In [ ]:
import matplotlib.pyplot as plt

def plot_metric(metric, **kwargs):
    plt.plot(range(len(metric)), metric, **kwargs)
    plt.xlabel("Epochs")
    plt.grid(True)


plot_metric(train_loss, marker="x")
plot_metric(eval_loss)
plt.ylabel("Loss")

In [47]:
import torch
from torch.optim import SGD
from torch.nn import Linear
from copy import deepcopy
from torch.optim import Optimizer

def average_optimizers(opts: list[Optimizer]) -> None:
    """
    Averages all state values across the given optimizers, inplace.

    Args:
        opts (list[Optimizer]): A list of PyTorch optimizers. All optimizers must have the same state structure.
    """
    if not opts:
        raise ValueError("The list of optimizers is empty.")
    
    states = [flatten_dict(opt.state) for opt in opts]


    for x in zip(*states):
        print(x)
        mean = torch.mean(torch.stack(x), dim=0)
        
        for state in x:
            state[:] = mean

def flatten_dict(d: dict) -> list:
    """
    Flattens an arbitrarily nested dictionary into a list of values.

    Args:
        d (Dict[Any, Any]): The dictionary to flatten.

    Returns:
        List[Any]: A list containing all the values in the dictionary, flattened.
    """
    values = []
    for v in d.values():
        if isinstance(v, dict):
            values.extend(flatten_dict(v))
        else:
            values.append(v)
    return values

# Define the average_optimizers function (paste your implementation here)

def test_average_optimizers_with_step():
    # Create a simple model
    model = Linear(10, 2)
    params = list(model.parameters())

    # Create synthetic data and a loss function
    x = torch.randn(5, 10)  # 5 samples, 10 features
    y = torch.randint(0, 2, (5,))  # 5 labels (binary classification)
    criterion = torch.nn.CrossEntropyLoss()

    # Create three optimizers with the same parameters
    optimizers = [
        SGD(params, lr=0.01, momentum=0.9),
        SGD(params, lr=0.01, momentum=0.9),
        SGD(params, lr=0.01, momentum=0.9)
    ]

    # Perform a single optimization step for each optimizer to populate momentum buffers
    for i, opt in enumerate(optimizers):
        opt.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        opt.step()

    # # Save the original states for comparison
    # original_states = [deepcopy(opt.state_dict()) for opt in optimizers]
    
    # print("Before Averaging:")
    # for i, opt in enumerate(optimizers):
    #     print(f"Optimizer {i+1} momentum buffers:")
    #     for param in params:
    #         if 'momentum_buffer' in opt.state[param]:
    #             print(opt.state[param]['momentum_buffer'])
    #         else:
    #             print("No momentum buffer.")

    # Average the optimizers
    average_optimizers(optimizers)

    # print("\nAfter Averaging:")
    # for i, opt in enumerate(optimizers):
    #     print(f"Optimizer {i+1} momentum buffers:")
    #     for param in params:
    #         if 'momentum_buffer' in opt.state[param]:
    #             print(opt.state[param]['momentum_buffer'])
    #         else:
    #             print("No momentum buffer.")

    # # Verify the results
    # for param in params:
    #     for state_key in original_states[0]['state'][param]:
    #         if isinstance(original_states[0]['state'][param][state_key], torch.Tensor):
    #             averaged_value = torch.mean(torch.stack([
    #                 original_states[i]['state'][param][state_key] for i in range(len(optimizers))
    #             ]), dim=0)

    #             for opt in optimizers:
    #                 assert torch.allclose(opt.state[param][state_key], averaged_value), \
    #                     f"State key '{state_key}' was not correctly averaged."

    # print("\nTest passed! All momentum buffers were averaged correctly.")

# Run the test
# test_average_optimizers_with_step()


In [50]:
import itertools

import numpy as np 
model = Linear(10, 2)
params = list(model.parameters())

# Create synthetic data and a loss function
x = torch.randn(5, 10)  # 5 samples, 10 features
y = torch.randint(0, 2, (5,))  # 5 labels (binary classification)
criterion = torch.nn.CrossEntropyLoss()

# Create three optimizers with the same parameters
optimizers = [
    SGD(params, lr=0.01, momentum=0.9),
    SGD(params, lr=0.01, momentum=0.9),
    SGD(params, lr=0.01, momentum=0.9)
]
for i, opt in enumerate(optimizers):
    opt.zero_grad()
    outputs = model(x)
    loss = criterion(outputs, y)
    loss.backward()
    opt.step()
op = optimizers[0]


# for v in op.state.values():
#     print("Value: ", type(v), v)
#     print("##################")

true_states = []
# for state in zip(*[opt.state.values() for opt in optimizers]):
#     print(type(state[0]))

# [opt.state.values() for opt in optimizers]
# for i in op.state.values():
#     for v in i.values():
#         print(v)

states = [[v for i in opt.state.values() for v in i.values()] for opt in optimizers]


# for x in zip(*states):
#     print(x)
#     mean = torch.mean(torch.stack(x), dim=0)
    
#     for state in x:
#         state[:] = mean

#     print(x)  
#     print("####")  
states[0]

[tensor([[ 0.8735,  0.2445,  0.6802, -0.2996,  0.0273, -0.1413,  0.1786, -0.1830,
           0.0087, -0.1419],
         [-0.8735, -0.2445, -0.6802,  0.2996, -0.0273,  0.1413, -0.1786,  0.1830,
          -0.0087,  0.1419]]),
 tensor([-0.1075,  0.1075])]